In [33]:
import random
import csv

def generate_tree(N_nodes=10000, percentage_cat=0.2, output_file="tree.csv"):
    # Ensure root exists
    parents = [-1]  # root has no parent

    # Probabilities for each node to be selected as a parent
    # This approximates an average number of children
    # by randomly assigning parents from earlier nodes.
    for node_id in range(1, N_nodes):
        parent_id = random.randint(0, max(0, node_id - 1))
        parents.append(parent_id)

    # Initialize categories with -1
    categories = [-1] * N_nodes

    # How many special categories?
    num_special = int(N_nodes * percentage_cat)

    # Select random nodes excluding the root
    special_nodes = random.sample(range(1, N_nodes), num_special)

    # Assign unique categories from 0–9999
    unique_classes = random.sample(range(N_nodes), num_special)

    for node, cls in zip(special_nodes, unique_classes):
        categories[node] = cls

    # Write CSV
    with open(output_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["node_id", "parent_id", "category"])
        for i in range(N_nodes):
            writer.writerow([i, parents[i], categories[i]])

    print(f"Generated tree with {N_nodes} nodes and saved to {output_file}")




In [59]:

generate_tree(
    N_nodes=10000,
    percentage_cat=0.001,
    output_file="tree.csv"
)


Generated tree with 10000 nodes and saved to tree.csv


# Solve

In [35]:
total_checks = 0



import numpy as np

tree = np.genfromtxt('shqiperi67.csv', delimiter=',', dtype=int, skip_header=1)

ids, parents, categories = tree[:,0], tree[:,1], tree[:,2]

from functools import lru_cache as cache


@cache(maxsize=None)
def solve_recursive(id):
    global total_checks
    total_checks += 1
    if categories[id] != -1:
        return categories[id]

    parent_id = parents[id]
    return solve_recursive(parent_id)


solved_categories = np.zeros(categories.shape)
for i in range(len(ids)):
    solved_categories[i] = solve_recursive(i)

In [36]:
a, b = np.unique(solved_categories, return_counts=True)
print(a, b)
most_common_category = a[np.argmax(b)]

print("Most common category:", most_common_category, "with count:", np.max(b))

[4.000e+00 7.000e+00 1.000e+01 ... 9.979e+03 9.995e+03 9.999e+03] [ 6  1  2 ...  2  1 39]
Most common category: 7728.0 with count: 9867


In [29]:
# see cache data

cache_info = solve_recursive.cache_info()
print(f"Cache info: {cache_info}")

Cache info: CacheInfo(hits=17042, misses=29435, maxsize=None, currsize=29435)


In [66]:


@cache
def maxProd(n):
    # Base cases
    if (n == 0 or n == 1):
        return 0
 
    max_val = 0
    for i in range(1, n - 1):
        last_max = max_val
        simple_split = i * (n - i)
        sub_split = maxProd(n - i) * i

        max_val = max(last_max, simple_split, sub_split)
 
    return max_val


maxProd(30)

59049

In [11]:
def knapsack(n, capacity, effort, skill):
    if n == 0 or capacity <= 0:
        return 0, []
    
    # if the weight of the nth item is more than capacity, then
    # this item cannot be included in the optimal solution
    if effort[n-1] > capacity:
        return knapsack(n-1, capacity, effort, skill)
    

    # Option 1: not take
    val1, idx1 = knapsack(n-1, capacity, effort, skill)
    # Option 2: take
    val2, idx2 = knapsack(n-1, capacity - effort[n-1], effort, skill)

    val2 += skill[n-1]

    idx2 = idx2 + [n-1]  # add current item index
    
    if val1 > val2:
        return val1, idx1
    else:
        return val2, idx2
 
import numpy as np
# read from csv file
names = []
effort_needed = []
skill_level = []

dat = np.genfromtxt("firstyear_project_students.csv", delimiter=',', dtype=None, names=True, encoding=None)

for row in dat:
    names.append(str(row['name']))
    effort_needed.append(row['effort_needed'])
    skill_level.append(row['skill_level'])

n = len(effort_needed)
capacity = 150

value, indices = knapsack(n, capacity, effort_needed, skill_level)
print(f"Max value: {value}, Indices: {[names[i] for i in indices]}")

# # save the names and effort and skill level to a csv file
# with open("knapsack_data.csv", "w", newline="") as f:
#     writer = csv.writer(f)
#     writer.writerow(["name", "effort_needed", "skill_level"])
#     for i in range(n):
#         writer.writerow([names[i], effort_needed[i], skill_level[i]])

Max value: 297, Indices: ['Emil', 'Lau', 'Luca', 'Noah']


# Simplified Knapsack Solution

The function above is already quite simple! Here's an even more concise version:

In [25]:
def knapsack_simple(index : int, capacity_left: int, weights: list, values: list) -> tuple:
    # firstly, check if we are out of items or capacity
    if index < 0 or capacity_left <= 0:
        return 0, []


    skip_item = knapsack_simple(index-1, capacity_left, weights, values)
    # check if the weight of the current item is more than capacity left,
    # if it is, we skip this item
    if weights[index] > capacity_left:
        return skip_item


    # We have two options:
    # yes: take the item at index
    # no: skip the item at index


    # Take the item at index
    yes = knapsack_simple(index-1, capacity_left - weights[index], weights, values)
    yes = (yes[0] + values[index], yes[1] + [index])

    # Don't take the item at index
    no = skip_item


    # if taking the item is better,
    if yes[0] > no[0]:
        return yes
    
    # otherwise
    return no


print("\n=== Ultra-Simple Version ===")
val, items = knapsack_simple(n-1, capacity, effort_needed, skill_level)
print(f"Value: {val}, Students: {[names[i] for i in items]}")


=== Ultra-Simple Version ===
Value: 297, Students: ['Emil', 'Lau', 'Luca', 'Noah']


# Knapsack Function Breakdown

## Function Signature
```python
def knapsack(n, capacity, weights, prices):
```
- `n`: Number of items to consider (first n items from the lists)
- `capacity`: Remaining capacity of the knapsack
- `weights`: List of item weights
- `prices`: List of item values/prices
- Returns: `(total_value, list_of_selected_item_indices)`

## Base Cases
```python
if n == 0 or capacity == 0:
    return 0, []
```
- If no items left to consider (`n == 0`) OR no capacity left (`capacity == 0`)
- Return 0 value and empty list (no items selected)

## Item Too Heavy Case
```python
if weights[n-1] > capacity:
    return knapsack(n-1, capacity, weights, prices)
```
- If the current item (item `n-1`) is too heavy for remaining capacity
- Skip it and solve for the remaining `n-1` items with same capacity

## Main Decision Logic
```python
else:
    # Option 1: Don't take current item
    val1, idx1 = knapsack(n-1, capacity, weights, prices)
    
    # Option 2: Take current item
    val2, idx2 = knapsack(n-1, capacity - weights[n-1], weights, prices)
    val2 += prices[n-1]  # Add value of current item
    idx2 = idx2 + [n-1]  # Add current item index to selected list
    
    # Return the better option
    if val1 > val2:
        return val1, idx1
    else:
        return val2, idx2
```

### Option 1: Don't Take Item
- Solve knapsack for remaining `n-1` items with same capacity
- Keep the value and indices as-is

### Option 2: Take Item
- Solve knapsack for remaining `n-1` items with reduced capacity
- Add the current item's value to the result
- Add the current item's index to the selected items list

### Choose Better Option
- Compare the two options and return the one with higher value
- If values are equal, it returns the second option (taking the item)

## Example Walkthrough
For `n=3, capacity=5, weights=[2,3,4], prices=[3,4,5]`:

1. Consider item 2 (weight=4, value=5)
2. Too heavy (4 > 5)? No, so consider both options:
   - Don't take: solve for n=2, capacity=5
   - Take: solve for n=2, capacity=1, then add value 5

This recursive approach explores all possible combinations but is exponential time complexity O(2^n).